# Module 01 — Why agents, why ADK

> **⚡ Quick path** — this is one of four modules on the 1-hour course preview.
> See the [README's Quick path section](../README.md#-quick-path---1-hour) for the sequence.

In this notebook you'll build two versions of the world's simplest agent — one without tools, one with a tool — and watch the **event stream** that connects the LLM, the tool, and you.

**What you'll leave with:** a mental model for the four primitives ADK gives you (`Agent`, `Runner`, `Event`, `Session`). Every later module unpacks one of those four.

**Runs in:** Google Colab or a local Python 3.10+ environment.
**Running cost:** well under $0.01 on OpenRouter's cheap models.


# Setup

Install dependencies, configure authentication, and import ADK.

## Install Dependencies

Run the cell below to install everything this module needs.

In [ ]:
!pip install -q google-adk==1.28.0 litellm==1.83.4 python-dotenv==1.0.1 nest-asyncio==1.6.0 deprecated==1.2.18 2>/dev/null

print("✅ Packages installed.")

## API Key Configuration

Part 1 of this course (M01–M10) uses **OpenRouter** so every demo works against Claude, GPT, Gemini, Qwen, or Gemma with a one-line swap. You need an `OPENROUTER_API_KEY`. Get one at [openrouter.ai/keys](https://openrouter.ai/keys) — pay-as-you-go, this notebook costs under a cent.

Two ways to provide it:

**Method 1 (Colab, recommended):** Click the 🔑 icon in the left sidebar → Add new secret → Name: `OPENROUTER_API_KEY` → Value: your key → enable notebook access.

**Method 2 (local or fallback):** Put it in a `.env` file next to the notebook, or paste it when prompted.

In [ ]:
import os

# Try Colab secrets first, then .env, then prompt.
OPENROUTER_API_KEY = None

try:
    from google.colab import userdata
    OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY")
    print("✅ API key loaded from Colab secrets.")
except Exception:
    try:
        from dotenv import load_dotenv
        load_dotenv()
        OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
        if OPENROUTER_API_KEY:
            print("✅ API key loaded from .env file.")
    except ImportError:
        pass

if not OPENROUTER_API_KEY:
    from getpass import getpass
    print("💡 Tip for Colab users: Go to 🔑 (left sidebar) → Add new secret → Name: OPENROUTER_API_KEY")
    OPENROUTER_API_KEY = getpass("Enter your OpenRouter API key: ")

assert OPENROUTER_API_KEY and OPENROUTER_API_KEY.strip(), "❌ No API key provided."
os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY

# Cheap + fast model for all vendor-agnostic demos. Swap it to try another provider.
MODEL_STRING = "openrouter/google/gemini-2.5-flash-lite"
print(f"✅ Model: {MODEL_STRING}")

## Import Libraries

ADK is Google's framework, so it natively speaks Gemini. To use it with any other model, we go through **LiteLLM** — a translation layer that turns one API format into many providers' formats. The `LiteLlm` wrapper you'll see below is the single line that makes ADK vendor-neutral.

In [ ]:
# Silence harmless import-time warnings before heavy imports.
import warnings, io, contextlib
warnings.filterwarnings("ignore")

with contextlib.redirect_stderr(io.StringIO()):
    import litellm
    litellm.suppress_debug_info = True
    import logging
    logging.getLogger("LiteLLM").setLevel(logging.WARNING)

    # Core ADK
    from google.adk.agents import LlmAgent
    from google.adk.runners import Runner
    from google.adk.sessions import InMemorySessionService
    from google.adk.models.lite_llm import LiteLlm
    from google.genai import types

import asyncio
import uuid

print("✅ Imports successful.")

# The four primitives

Most tutorials throw a dozen concepts at you in the first module. ADK only has four that matter on day one.

| Primitive | What it is | What you do with it |
|---|---|---|
| `LlmAgent` | An LLM wired to instructions, a model, and (optionally) tools | You define one |
| `Runner` | The event loop that drives a conversation | You call `.run_async()` |
| `Event` | Every message, tool call, tool response, and state change | You read them to see what's happening |
| `Session` | The conversation's memory — event history plus a state dict | You create one per conversation |

Everything else — workflow agents, multi-agent hierarchies, callbacks, memory services, evaluation — composes on top of these four. Keep them in mind; they'll come back every module.

# The World's Simplest Agent

Four required arguments: a name, a model, a description, and an instruction. No tools yet — this is just an LLM with a system prompt, wrapped in ADK's plumbing.

In [ ]:
greeter = LlmAgent(
    name="greeter",
    model=LiteLlm(model=MODEL_STRING),
    description="Greets the user in a friendly way.",
    instruction="You are a friendly greeter. Respond in one short sentence.",
)

print(f"✅ Agent '{greeter.name}' built.")

## Running the Agent

An `LlmAgent` on its own doesn't do anything. It needs a `Runner` to drive it, and a `Session` to hold the conversation. `runner.run_async(...)` returns an **event stream** — one yielded event per step. Watch the stream carefully; this is how ADK makes agent behavior visible.

In [ ]:
APP = "m01_demo"
USER = "student"

session_service = InMemorySessionService()

async def chat(agent, prompt: str):
    """Send one user message to the agent and print every event it emits."""
    # Fresh session per call so previous runs don't leak in.
    sid = f"session-{uuid.uuid4().hex[:8]}"
    await session_service.create_session(app_name=APP, user_id=USER, session_id=sid)
    runner = Runner(agent=agent, app_name=APP, session_service=session_service)

    message = types.Content(role="user", parts=[types.Part(text=prompt)])

    print(f"USER: {prompt}\n")
    async for event in runner.run_async(user_id=USER, session_id=sid, new_message=message):
        tag = "[FINAL]" if event.is_final_response() else "[step]"
        if event.content and event.content.parts:
            for p in event.content.parts:
                if p.text:
                    print(f"{tag} {event.author}: {p.text.strip()}")
                if p.function_call:
                    print(f"[tool_call] {p.function_call.name}({dict(p.function_call.args)})")
                if p.function_response:
                    print(f"[tool_resp] {p.function_response.response}")

await chat(greeter, "Hi, what's your name?")

One event came back, containing the final text response. That's the minimum a conversation can produce — a user turn goes in, one model turn comes out, the stream ends. **Events are the unit of observability in ADK:** every message, tool call, tool response, state change, or agent hand-off is one. Later modules add tools, delegation, and state mutations; you'll see the event stream grow.

# Add a Tool — And Watch the Events Multiply

Tools are Python functions with a **docstring** and **type hints**. ADK reads those to build a JSON schema the model can see. The model decides to call the tool; ADK executes it; the result comes back as a tool-response event; the model produces a final answer.

You'll see three events instead of one:

```
user turn → [tool_call] → [tool_resp] → [FINAL] text response
```

In [ ]:
def get_weather(city: str) -> dict:
    """Look up today's weather for a city.

    Use this tool whenever the user asks about weather. The input is a city name
    as a string; the output is a dict with 'city' and 'report' keys.
    """
    fake_db = {
        "Bratislava": "Sunny, 18°C",
        "Prague": "Cloudy, 14°C",
        "Munich": "Rainy, 11°C",
    }
    return {
        "city": city,
        "report": fake_db.get(city, f"No data for {city}."),
    }

weather_agent = LlmAgent(
    name="weather_agent",
    model=LiteLlm(model=MODEL_STRING),
    description="Reports weather for a given city.",
    instruction=(
        "You are a weather assistant. When the user asks about weather in a city, "
        "call the get_weather tool and report what it returns. Be brief."
    ),
    tools=[get_weather],
)

await chat(weather_agent, "What's the weather in Prague?")

## What Just Happened

The output above is the smallest complete picture of an agent run:

1. **`[tool_call] get_weather({'city': 'Prague'})`** — the model decided a tool was needed and emitted a structured call. ADK caught it before it left the agent.
2. **`[tool_resp] {'city': 'Prague', 'report': 'Cloudy, 14°C'}`** — ADK executed the Python function and fed the return value back to the model as a tool-response event.
3. **`[FINAL] weather_agent: ...`** — the model wrote a natural-language answer using the tool's output.

Three events, all visible, all inspectable. If the model had called the tool wrong, you'd see the error. If it had refused to call the tool, you'd see the refusal. If it had called two tools in sequence, you'd see both. This visibility is ADK's single biggest pedagogical advantage over stringing together raw LLM calls — you can read the agent's reasoning by reading the events.

## A Word on `adk web`

Everything you just printed can be browsed visually by running `adk web` from a folder containing your agent. It opens a chat UI with a live event timeline beside the conversation — the same events, clickable, inspectable, with the full JSON payloads. If you're learning ADK, run `adk web` often. We'll rely on the text printouts in this course because they drop into a video recording cleanly, but the web UI is the best debugger you get for free.

# Your Turn

Three small changes, five minutes each. Run them in a new cell below so you can diff the event streams.

1. **Change the model.** Replace `MODEL_STRING` with `openrouter/anthropic/claude-haiku-4-5` or `openrouter/openai/gpt-4o-mini`. Re-run the weather agent. Does the final answer change? Do the events change shape?
2. **Break the tool on purpose.** Ask the weather agent about `"Reykjavik"` (not in the fake database). Read the events. How does the model handle the `"No data for Reykjavik."` response — does it pass the report through honestly, or hallucinate?
3. **Add a second tool.** Write a `convert_celsius_to_fahrenheit(celsius: float) -> float` tool, add it to the agent, and ask *"What's the weather in Munich in Fahrenheit?"*. Watch the event stream — how many tool calls does the model make? In what order?

No grading. The goal is to get a feel for what shows up in the event stream and what doesn't.

# Key Takeaways

- An ADK agent is a configuration object. Four required arguments — name, model, description, instruction — plus an optional list of tools.
- The **Runner** drives conversations forward; `.run_async()` yields an **Event** per step.
- Every meaningful moment in an agent run is an Event: user input, tool calls, tool responses, state changes, final text. Read events to debug.
- The **`LiteLlm` wrapper** is the one-line swap that makes ADK vendor-neutral. The same agent runs on Gemini, Claude, GPT, or any OpenRouter-routable model.
- The four primitives to carry forward: **`Agent` · `Runner` · `Event` · `Session`**. Every later module composes on top of them.

# Next up — M02: Tools as verbs

You just saw one flavor of tool: a plain Python function. ADK supports three more — an OpenAPI spec, an MCP server, and another agent wrapped as a tool. M02 builds one of each. Before you move on, make sure you can say out loud what each primitive is. If not, re-read the event-stream output once more.